# Oracle DBA Management

Copyright 2026, Denis Rothman

*Note: Version 2 focuses on Schema Initialization for section 3.3 Initialize HR Recruitment Schema*

This notebook is designed as a foundational management script for an Oracle DBA, specifically focused on setting up and maintaining **Vector Tables** in an Oracle 23ai environment.

It automates the lifecycle of vector storage, from connection setup to schema creation and data sanitation, providing a reusable template for managing AI-driven database schemas.

### Notebook Summary: Oracle Vector Table Management

* **Environment Setup:** Configures the Python environment by installing the necessary `oracledb` driver and managing secure wallet credentials. It handles the mounting of Google Drive to access configuration files and Oracle Wallets securely.
* **Database Connection:** Establishes a secure connection to the Oracle database using the thick or thin client mode via the Oracle Wallet, ensuring secure access with user credentials.
* **Schema Initialization (DDL):** Defines and creates two specialized vector tables, `CONTEXT_LIBRARY` and `KNOWLEDGE_STORE`. These tables are optimized for RAG (Retrieval-Augmented Generation) applications, storing high-dimensional vectors (`VECTOR(1536)`) alongside textual metadata.
* **Schema Initialization for 3.3 Initialize HR Recruitment Schema
* **Verification & Auditing:** Includes diagnostic queries to query `user_tables` and `user_tab_columns`. This verifies the successful creation of tables and confirms the correct data types (e.g., `VECTOR`, `CLOB`) are applied.
* **Data Maintenance (Sanitation):** Implements administrative functions to `TRUNCATE` tables. This allows the DBA to rapidly clear vector data for testing or resetting environments without dropping the table structures.
* **Health Checks:** Provides a utility to verify table status, reporting row counts and sampling content to ensure operations (like truncation) were successful and the tables are ready for new data ingestion.


# 1.Installation and Setup

## DBA Parameters



## 1.1 Global Configuration Flags

This section acts as the **Control Panel** for the notebook. These boolean flags determine which administrative tasks will execute during this run. This allows the DBA to run the notebook safely in "maintenance mode" without accidentally recreating tables or wiping data.

* **`unzip_wallet`**: Set to `True` only for the initial setup to extract the Oracle Wallet configuration. Once extracted, set to `False` to avoid redundant file operations.
* **`create_tables`**: Set to `True` to execute DDL statements. This initializes the `CONTEXT_LIBRARY` and `KNOWLEDGE_STORE` tables. (Keep `False` if tables already exist).
* **`empty_tables`**: Set to `True` to perform a **TRUNCATE** operation. **Warning:** This will permanently delete all vector data from the tables while preserving the structure.

In [10]:
unzip_wallet=False  # True to unzip the wallet. False to only unzip the wallet once
create_tables=True # True to create tables
empty_tables=False  # True to empty tables (TRUNCATE)
drop_tables=False   # True to DROP all tables (Clean Slate)

## 1.2 Oracle environment Setup & Wallet Extraction

This step prepares the runtime environment by connecting to Google Drive and ensuring the Oracle Wallet is available. The Oracle Wallet contains the SSL/TLS certificates and configuration files (tnsnames.ora, sqlnet.ora) necessary for a secure connection to the Oracle Autonomous Database.

Google Drive Mount: Maps your personal Drive to /content/drive, allowing the notebook to read credentials and configuration files stored externally.

Wallet Unzipping: If unzip_wallet is set to True, the script searches for the wallet ZIP file in the specified path and extracts it. This ensures the connection drivers can locate the required security certificates.

Path Definition: Sets wallet_path to the directory where the unzipped configuration files reside, which will be passed to the Oracle driver in subsequent steps.

In [11]:
from google.colab import drive
drive.mount('/content/drive')
if unzip_wallet==True:
  !unzip -o "/content/drive/MyDrive/files/oracle/Wallet_*.zip" -d "/content/drive/MyDrive/files/oracle"
wallet_path = "/content/drive/MyDrive/oracle_wallet"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.3 Install Oracle Database Driver
This command installs the oracledb Python driver, which is the renamed and modernized successor to the legacy cx_Oracle driver. It serves as the bridge between this Python notebook and the Oracle Database.

Version Pinning (==3.4.1): The version is explicitly fixed to 3.4.1 to ensure stability and reproducibility. In production DBA scripts, pinning versions prevents unexpected updates or API changes from breaking the automation workflow.

Thin Client Mode: By default, this driver operates in "Thin" mode, meaning it communicates directly with the database using pure Python code without requiring local Oracle Client libraries (Instant Client).

In [12]:
!pip install oracledb==3.4.1

In [13]:
import oracledb
print(oracledb.__version__)

3.4.1


## 1.4. Additional imports

In [14]:
# Imports for this notebook
import json
import time
from tqdm.auto import tqdm
import tiktoken                                                         # tokenization
from tenacity import retry, stop_after_attempt, wait_random_exponential # to retry functions
import re
import textwrap
from IPython.display import display, Markdown
import copy

# 2.Establish Secure Database Connection
This step handles the authentication and connection to the Oracle 23ai instance. It is designed to adhere to security best practices by strictly separating code from credentials.

External Credential Management: Instead of hardcoding sensitive passwords directly into the notebook (which is a security risk), the script reads a credentials.txt file stored securely on Google Drive.

Credential Parsing: The read_key_value_file helper function parses the external file to retrieve the username, password, Wallet password, and DSN (Data Source Name).

Connection Initialization: The oracledb.connect() method establishes the session using the extracted credentials and the local Wallet configuration path.

Connectivity Test: A simple "Heartbeat" query (SELECT ... FROM DUAL) is executed immediately to verify that the connection is active and stable before proceeding.

In [15]:
import os
from google.colab import userdata
oracle_dsn = userdata.get('oracle_dsn')

creds_path = "/content/drive/MyDrive/files/oracle/credentials.txt"
if not os.path.exists(creds_path):
    raise FileNotFoundError(f"Credentials file not found: {creds_path}")

def read_key_value_file(path):
    creds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            creds[k.strip()] = v.strip()
    return creds

creds = read_key_value_file(creds_path)

# Use values (do not print them)
user = creds.get("user")
password = creds.get("password")
wallet_password = creds.get("wallet_password")
dsn = creds.get("dsn", oracle_dsn)
wallet_path = creds.get("wallet_path", "/content/drive/MyDrive/files/oracle")

import oracledb
connection = oracledb.connect(
    user=user,
    password=password,
    dsn=dsn,
    config_dir=wallet_path,
    wallet_location=wallet_path,
    wallet_password=wallet_password
)

cursor = connection.cursor()
cursor.execute("SELECT 'Connected!' FROM dual")
print(cursor.fetchone())

('Connected!',)


# 3 Creating the tables

## 3.1 Initialize Vector Schema

This section executes the Data Definition Language (DDL) commands to build the database schema. The code relies on the create_tables flag to prevent accidental overwrites of existing structures.

Two specialized tables are created to support the AI application:

CONTEXT_LIBRARY: Stores structured blueprints and metadata.

embedding VECTOR(1536): This column type is specific to Oracle 23ai. It is optimized to store high-dimensional vectors (compatible with OpenAI's text-embedding-3-small or ada-002 models, which use 1536 dimensions).

blueprint_json CLOB: Uses a Character Large Object (CLOB) to store potentially large JSON structures.

KNOWLEDGE_STORE: Stores unstructured text segments for retrieval.

It links raw text (text CLOB) directly with its vector representation for efficient similarity searches.

In [16]:
import oracledb

# Create the vector tables in Oracle 23ai (Look Before You Leap method)
if create_tables == True:

    # Define our tables and their DDL statements in a dictionary
    tables_to_create = {
        "context_library": """
            CREATE TABLE context_library (
                id VARCHAR2(200) PRIMARY KEY,
                description CLOB,
                blueprint_json CLOB,
                embedding VECTOR(1536)
            )
        """,
        "knowledge_store": """
            CREATE TABLE knowledge_store (
                id VARCHAR2(200) PRIMARY KEY,
                source VARCHAR2(200),
                text CLOB,
                embedding VECTOR(1536)
            )
        """
    }

    # Loop through and check existence before creating
    for table_name, ddl in tables_to_create.items():

        # 1. Check if the table already exists in user_tables (Must be UPPERCASE!)
        check_sql = f"SELECT count(*) FROM user_tables WHERE table_name = '{table_name.upper()}'"
        cursor.execute(check_sql)
        exists = cursor.fetchone()[0]

        # 2. If count is 0, the table does not exist, so we create it
        if exists == 0:
            print(f"🔨 Creating table '{table_name}'...")
            cursor.execute(ddl)
            print(f"✅ Table '{table_name}' created successfully.")
        # 3. If count is 1, we skip it safely
        else:
            print(f"⚠️ Table '{table_name}' already exists. Skipping creation.")

    connection.commit()

⚠️ Table 'context_library' already exists. Skipping creation.
⚠️ Table 'knowledge_store' already exists. Skipping creation.


## 3.2 Schema Audit & Verification

After attempting to create the tables, this step performs a structural audit to confirm that the database schema matches the intended design. It queries the Oracle Data Dictionary views (user_tables and user_tab_columns) rather than just assuming the CREATE command worked.

Key verification points include:

Table Existence: Confirms that CONTEXT_LIBRARY and KNOWLEDGE_STORE are present in the user's schema.

Data Type Validation: Specifically checks that the EMBEDDING columns are strictly defined as VECTOR types (rather than generic BLOBs or arrays), which is critical for enabling vector similarity search functions later.

Connection Health: Runs a final query on the DUAL table to ensure the session remains active and responsive after the DDL operations.

In [17]:
# Verify that the tables exist and the VECTOR columns are correct

print("Checking Oracle vector tables...\n")

cursor.execute("""
SELECT table_name
FROM user_tables
WHERE table_name IN ('CONTEXT_LIBRARY', 'KNOWLEDGE_STORE')
""")

tables = cursor.fetchall()
print("Tables found:", tables)

print("\nColumn definitions for CONTEXT_LIBRARY:")
cursor.execute("""
SELECT column_name, data_type, data_length
FROM user_tab_columns
WHERE table_name = 'CONTEXT_LIBRARY'
ORDER BY column_id
""")
for row in cursor.fetchall():
    print(row)

print("\nColumn definitions for KNOWLEDGE_STORE:")
cursor.execute("""
SELECT column_name, data_type, data_length
FROM user_tab_columns
WHERE table_name = 'KNOWLEDGE_STORE'
ORDER BY column_id
""")
for row in cursor.fetchall():
    print(row)

print("\nRunning a simple test query on DUAL...")
cursor.execute("SELECT 'Oracle connection OK' FROM dual")
print(cursor.fetchone()[0])


Checking Oracle vector tables...

Tables found: [('CONTEXT_LIBRARY',), ('KNOWLEDGE_STORE',)]

Column definitions for CONTEXT_LIBRARY:
('ID', 'VARCHAR2', 200)
('DESCRIPTION', 'CLOB', 4000)
('BLUEPRINT_JSON', 'CLOB', 4000)
('EMBEDDING', 'VECTOR', 8200)

Column definitions for KNOWLEDGE_STORE:
('ID', 'VARCHAR2', 200)
('SOURCE', 'VARCHAR2', 200)
('TEXT', 'CLOB', 4000)
('EMBEDDING', 'VECTOR', 8200)

Running a simple test query on DUAL...
Oracle connection OK


##  3.3 Initialize HR Recruitment Schema (Chapter 3: Hybrid RAG)

In [18]:
# -------------------------------------------------------------------------
# 3.3 Initialize HR Recruitment Schema (Chapter 3: Hybrid RAG)
# -------------------------------------------------------------------------
# This schema demonstrates the "Converged Database" pattern.
# Unlike the KNOWLEDGE_STORE (which is just text chunks), the CANDIDATE_POOL
# contains structured SQL columns (Salary, Years Experience) alongside vectors.

if create_tables == True:
    print("\n--- Creating HR Recruitment Schema ---")

    # 1. CANDIDATE_POOL: The Hybrid Table (Structured + Vector)
    # We add constraints (CHECK constraints) to show real DBA work.
    cursor.execute("""
    CREATE TABLE candidate_pool (
        candidate_id VARCHAR2(50) PRIMARY KEY,
        full_name VARCHAR2(100),
        summary CLOB,                   -- The text we will vectorise
        skills VARCHAR2(1000),          -- Comma-separated list for keywords
        years_experience NUMBER,        -- For SQL Filtering (e.g. > 5 years)
        salary_expectation NUMBER,      -- For SQL Filtering (e.g. < 120k)
        resume_vector VECTOR(1536)      -- The Semantic Brain
    )
    """)
    print("Table CANDIDATE_POOL created.")

    # 2. RECRUITMENT_RULES: Domain-Specific Instructions
    # We separate this from the generic CONTEXT_LIBRARY to show domain isolation.
    cursor.execute("""
    CREATE TABLE recruitment_rules (
        rule_id VARCHAR2(50) PRIMARY KEY,
        agent_persona CLOB,             -- e.g., "Culture Fit Officer" vs "Technical Screener"
        evaluation_criteria CLOB,       -- The specific rubric for this agent
        rule_vector VECTOR(1536)
    )
    """)
    print("Table RECRUITMENT_RULES created.")

    connection.commit()
    print("HR Schema initialized successfully.")


--- Creating HR Recruitment Schema ---
Table CANDIDATE_POOL created.
Table RECRUITMENT_RULES created.
HR Schema initialized successfully.


## 3.4 Verification of Converged Schema

In [19]:
# 3.4 Verification of Converged Schema
# -------------------------------------------------------------------------
# This cell verifies that both the legacy Chapter 2 tables (Document Store)
# and the new Chapter 3 tables (HR Application) coexist in the database.

print("Checking full schema (Chapter 2 + Chapter 3 tables)...\n")

# 1. Check existence of ALL tables
cursor.execute("""
SELECT table_name
FROM user_tables
WHERE table_name IN ('CONTEXT_LIBRARY', 'KNOWLEDGE_STORE', 'CANDIDATE_POOL', 'RECRUITMENT_RULES')
ORDER BY table_name
""")

tables = cursor.fetchall()
print(f"Tables found ({len(tables)}/4):", tables)

# 2. Inspect the "Hybrid" Table (CANDIDATE_POOL)
# Look specifically for the mix of relational columns (SALARY, EXPERIENCE) and VECTOR columns.
print("\n--- Column definitions for CANDIDATE_POOL (Hybrid Table) ---")
cursor.execute("""
SELECT column_name, data_type, data_length
FROM user_tab_columns
WHERE table_name = 'CANDIDATE_POOL'
ORDER BY column_id
""")
for row in cursor.fetchall():
    # Highlight the vector and scalar columns for visual confirmation
    print(row)

# 3. Inspect the Rules Table (RECRUITMENT_RULES)
print("\n--- Column definitions for RECRUITMENT_RULES ---")
cursor.execute("""
SELECT column_name, data_type, data_length
FROM user_tab_columns
WHERE table_name = 'RECRUITMENT_RULES'
ORDER BY column_id
""")
for row in cursor.fetchall():
    print(row)

# 4. Connection Heartbeat
print("\nRunning a simple test query on DUAL...")
cursor.execute("SELECT 'Oracle connection OK' FROM dual")
print(cursor.fetchone()[0])

Checking full schema (Chapter 2 + Chapter 3 tables)...

Tables found (4/4): [('CANDIDATE_POOL',), ('CONTEXT_LIBRARY',), ('KNOWLEDGE_STORE',), ('RECRUITMENT_RULES',)]

--- Column definitions for CANDIDATE_POOL (Hybrid Table) ---
('CANDIDATE_ID', 'VARCHAR2', 50)
('FULL_NAME', 'VARCHAR2', 100)
('SUMMARY', 'CLOB', 4000)
('SKILLS', 'VARCHAR2', 1000)
('YEARS_EXPERIENCE', 'NUMBER', 22)
('SALARY_EXPECTATION', 'NUMBER', 22)
('RESUME_VECTOR', 'VECTOR', 8200)

--- Column definitions for RECRUITMENT_RULES ---
('RULE_ID', 'VARCHAR2', 50)
('AGENT_PERSONA', 'CLOB', 4000)
('EVALUATION_CRITERIA', 'CLOB', 4000)
('RULE_VECTOR', 'VECTOR', 8200)

Running a simple test query on DUAL...
Oracle connection OK


# 4 Data maintenance

##4.1 Data Maintenance: Truncate Tables

This section defines the administrative function responsible for clearing vector data. It is controlled by the empty_tables flag defined in the Control Panel (Step 1.1).

TRUNCATE vs. DELETE: The function uses the SQL TRUNCATE command rather than DELETE. TRUNCATE is a Data Definition Language (DDL) operation that instantly resets the table's storage "High Water Mark." This is significantly faster and more resource-efficient than deleting rows individually, especially for tables containing complex VECTOR data types.

Operational Safety: The function is encapsulated in a try-except block to catch and report any locking issues or permission errors during the process.

In [20]:
def empty_vector_tables(cursor):
    """
    Empties ALL vector tables (Chapter 2 & Chapter 3) using TRUNCATE.
    TRUNCATE is faster than DELETE and resets storage.
    """
    try:
        # --- Chapter 2 Tables ---
        print("Emptying context_library...")
        cursor.execute("TRUNCATE TABLE context_library")

        print("Emptying knowledge_store...")
        cursor.execute("TRUNCATE TABLE knowledge_store")

        # --- Chapter 3 Tables (HR Schema) ---
        print("Emptying candidate_pool...")
        cursor.execute("TRUNCATE TABLE candidate_pool")

        print("Emptying recruitment_rules...")
        cursor.execute("TRUNCATE TABLE recruitment_rules")

        print("All tables successfully emptied.")

    except Exception as e:
        print(f"Error emptying tables: {e}")

# Usage example
if empty_tables == True:
    empty_vector_tables(cursor)

## 4.2 Validate Data Removal

This verification utility acts as a final quality gate, ensuring that the tables are in a "clean slate" state. It confirms the success of the truncation operation by querying the current state of the database.

Row Count Audit: It queries SELECT COUNT(*) for both tables to provide definitive proof that the row count is zero.

Visual Status Indicators: The script uses visual tags ([EMPTY] ✅ or [NOT EMPTY] ❌) to allow the DBA to instantly scan the output logs for success or failure.

Forensic Sampling: In the event that the tables are not empty (indicating a failure to truncate), the script automatically fetches and displays a sample of the remaining data (first 2 rows). This helps the DBA quickly diagnose why the data persists (e.g., if a rollback occurred or if the wrong table was targeted).

This completes the descriptions for the entire notebook. The notebook is now structured as a professional DBA tool with clear sections:

Installation & Setup (Control Panel, Environment, Driver)

Connection (Secure Login)

Schema Creation (Table DDL, Verification)

Maintenance (Truncate, Validation)

In [21]:
def verify_tables_empty(cursor):
    """
    Checks the row count and content of ALL vector tables
    to verify they are empty.
    """
    # Updated list to include HR tables
    tables = [
        "context_library",
        "knowledge_store",
        "candidate_pool",
        "recruitment_rules"
    ]

    print("--- Verifying Table Status ---")

    for table in tables:
        try:
            # 1. Get the Row Count
            # We use distinct queries because some tables might not exist if skipped
            cursor.execute(f"SELECT count(*) FROM user_tables WHERE table_name = UPPER('{table}')")
            exists = cursor.fetchone()[0]

            if exists:
                cursor.execute(f"SELECT COUNT(*) FROM {table}")
                row_count = cursor.fetchone()[0]

                print(f"\nTable: {table}")
                print(f"Row Count: {row_count}")

                # 2. Check Content
                if row_count == 0:
                    print("Status: [EMPTY] \u2705") # Green checkmark
                else:
                    print("Status: [NOT EMPTY] \u274C") # Red X
                    print("Sample Content (First 2 rows):")
                    cursor.execute(f"SELECT * FROM {table} FETCH FIRST 2 ROWS ONLY")
                    rows = cursor.fetchall()
                    for row in rows:
                        print(f" - Data snippet: {str(row)[:50]}...")
            else:
                 print(f"\nTable: {table} (Not found in schema)")

        except Exception as e:
            print(f"Error checking {table}: {e}")

    print("\n------------------------------")

# Usage
if empty_tables == True:
    verify_tables_empty(cursor)

## 4.3.Destructive Maintenance: Drop Schema (Start From Scratch)

In [22]:
# -------------------------------------------------------------------------
# 4.3 Destructive Maintenance: Drop Schema (Start From Scratch)
# -------------------------------------------------------------------------
# This section allows you to completely remove the schema objects.
# Use this if you want to reset the environment before re-running Section 3.
# -------------------------------------------------------------------------

if drop_tables == True:
    print("\n--- WARNING: Initiating Schema Teardown (Destructive) ---")

    # List tables in reverse dependency order (if applicable).
    # We use PURGE to skip the Recycle Bin and free space immediately.
    tables_to_drop = [
        "RECRUITMENT_RULES",
        "CANDIDATE_POOL",
        "KNOWLEDGE_STORE",
        "CONTEXT_LIBRARY"
    ]

    for table in tables_to_drop:
        try:
            print(f"Attempting to drop table {table}...", end=" ")
            cursor.execute(f"DROP TABLE {table} PURGE")
            print("✅ DROPPED.")
        except oracledb.Error as e:
            error_obj = e.args[0]
            # ORA-00942: table or view does not exist (Safe to ignore)
            if error_obj.code == 942:
                print("⏭️ SKIPPED (Does not exist).")
            else:
                print(f"❌ FAILED: {e}")

    print("\nTeardown complete. You can now re-run Section 3 to create fresh tables.")